In [ ]:
#| default_exp edit

## Notebook editing

`edit_notebook` is the production notebook mutation adapter. It keeps text transforms deterministic, applies notebook changes all-or-nothing, and returns structured diffs that MCP clients can inspect without parsing raw `.ipynb` JSON.

The shape is intentionally small: pure text helpers first, then one notebook/cell adapter. Read tools still provide context; this module only mutates notebooks.

In [ ]:
#| export
import ast,copy,difflib,re
from dataclasses import dataclass
from pathlib import Path
from fastcore.basics import patch
from fastcore.nbio import Notebook as FastcoreNotebook, mk_cell, read_nb
from nbskill.foundation import (
    Notebook,apply_directives,cap_text,cell_source,clear_outputs,commit_notebook,source_hash)
from nbskill.parallel import notebook_locks
from nbskill.write import append_notebook_edit_feedback

In [ ]:
#| export
_EDIT_MAX_OPERATIONS = 50
_EDIT_MAX_SOURCE_CHARS = 200000
_EDIT_MAX_AFFECTED_CELLS = 200
_EDIT_MAX_DIFF_CHARS = 50000
_EDIT_WARN_AFFECTED_CELLS = 5
_EDIT_WARN_CODE_LINES = 20
_EDIT_WARN_TOP_LEVEL_FUNCTIONS = 3

In [ ]:
#| export
def _edit_text_budget(value):
    if value is None: return 0
    if isinstance(value, str): return len(value)
    if isinstance(value, dict): return sum(_edit_text_budget(item) for item in value.values())
    if isinstance(value, (list, tuple)): return sum(_edit_text_budget(item) for item in value)
    return 0

In [ ]:
#| export
def _validate_edit_budget(edits):
    if len(edits) > _EDIT_MAX_OPERATIONS:
        raise ValueError(f"edit_notebook budget exceeded: {len(edits)} operations exceeds limit {_EDIT_MAX_OPERATIONS}")
    chars = _edit_text_budget(edits)
    if chars > _EDIT_MAX_SOURCE_CHARS:
        raise ValueError(f"edit_notebook budget exceeded: {chars} text chars exceeds limit {_EDIT_MAX_SOURCE_CHARS}")

Craft warnings make risky notebook edits visible without changing edit semantics. They now flag calls whose definitions remain below the edited cell, so a notebook stays executable from top to bottom.

## Text and line transforms

The first layer edits plain text. Keeping it independent from notebook I/O makes the same operations useful in direct Python calls and structured notebook edits.

In [ ]:
#| export
def _cell_directive_body_lines(source):
    lines = str(source or "").strip("\n").splitlines()
    directives, body_start = [], len(lines)
    for idx, line in enumerate(lines):
        stripped = line.strip()
        if not stripped: continue
        if not stripped.startswith("#|"):
            body_start = idx
            break
        directive = stripped[2:].strip()
        if not directive.startswith("default_exp"):
            directives.append(stripped)
    return directives, lines[body_start:]

In [ ]:
#| export
def _craft_warning(code, message, path, cell_id=None, **extra):
    warning = {"code": code, "message": message, "path": str(path), "severity": "warning", **extra}
    if cell_id: warning["cell_id"] = cell_id
    return warning

In [ ]:

#| export
def _code_content_lines(source):
    return [line for line in str(source or "").splitlines() if line.strip() and not line.lstrip().startswith("#|")]

In [ ]:
#| export
def _top_level_function_count(source):
    raw = str(source or "").strip("\n")
    if not raw: return 0
    _, body_lines = _cell_directive_body_lines(raw)
    try: tree = ast.parse("\n".join(body_lines))
    except SyntaxError: return 0
    return sum(1 for node in tree.body if _is_function_def(node))

In [ ]:
#| export
def _is_function_def(node): return isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))

In [ ]:
#| export
def _edit_notebook_craft_warnings(path, nb, affected_cell_ids):
    warnings, affected = [], set(affected_cell_ids)
    if len(affected_cell_ids) > _EDIT_WARN_AFFECTED_CELLS:
        warnings.append(_craft_warning(
            "many-affected-cells",
            f"This edit touched {len(affected_cell_ids)} cells; split it into smaller edit_notebook calls and run a focused check after each one.",
            path,
            affected_cells=len(affected_cell_ids)))
    for cell in nb.cells:
        cell_id = getattr(cell, "id", "")
        if cell_id not in affected or getattr(cell, "cell_type", None) != "code": continue
        source = cell_source(cell)
        line_count = len(_code_content_lines(source))
        if line_count > _EDIT_WARN_CODE_LINES:
            warnings.append(_craft_warning("large-code-cell",
                "This code cell is large; split this into markdown + implementation + example + test cells.",
                path,cell_id=cell_id,line_count=line_count))
        function_count = _top_level_function_count(source)
        if function_count > _EDIT_WARN_TOP_LEVEL_FUNCTIONS:
            warnings.append(_craft_warning("multi-function-cell",
                f"This code cell defines {function_count} top-level functions; try op=\"explode_cells\" on cell_id={cell_id!r}.",
                path,cell_id=cell_id,function_count=function_count))
    from nbskill.graph import notebook_order_problems
    for problem in notebook_order_problems(path, nb=nb):
        if problem["code"] != "cell-order" or problem["cell_id"] not in affected: continue
        warnings.append(_craft_warning(
            "cell-order", f"{problem['symbol']!r} {problem['detail']}; move it above this cell.", path,
            cell_id=problem["cell_id"], symbol=problem["symbol"], line=problem["line"]
        ))
    return warnings

In [ ]:
#| export
def _format_edit_warnings(warnings):
    if not warnings: return ""
    return "\n".join(f"- {warning['code']}: {warning['message']}" for warning in warnings)

In [ ]:
#| export
def _cap_edit_diffs(diffs): return [{**item, "diff": cap_text(item.get("diff", ""), _EDIT_MAX_DIFF_CHARS)} for item in diffs]

#### Pure text helpers

The text helpers do not know about notebooks. They take strings, return strings, and raise clear `ValueError`s for invalid line bounds or malformed replacements. That makes cell-level and notebook-level editing share the same behavior.

In [ ]:
#| export
def _coerce_lines(lines):
    if lines is None: return []
    if isinstance(lines, str): return lines.splitlines()
    out = []
    for line in lines:
        parts = str(line).splitlines()
        out.extend(parts if parts else [""])
    return out

In [ ]:
#| export
def _join_lines(lines): return chr(10).join(str(line) for line in lines)

In [ ]:

#| export
def _line_no(value, n_lines, name):
    try: value = int(value)
    except (TypeError, ValueError) as err: raise ValueError(f"{name} must be an integer") from err
    if value < 0: value = n_lines + value + 1
    return value

In [ ]:

#| export
def _norm_lines(text, start_line, end_line=None):
    """Return zero-based inclusive-exclusive indexes for 1-based line bounds."""
    lines = str(text).splitlines()
    n_lines = len(lines)
    if n_lines == 0: raise ValueError("cannot edit line ranges in an empty cell")
    start = _line_no(start_line, n_lines, "start_line")
    end = _line_no(start_line if end_line is None else end_line, n_lines, "end_line")
    if start < 1 or end < start or end > n_lines:
        raise ValueError(f"line bounds must be within 1:{n_lines}; got {start_line!r}:{end_line!r}")
    return start - 1, end

In [ ]:
#| export
def _unified_diff(before, after, fromfile="before", tofile="after", context=2):
    """Return a compact unified diff, or an empty string when text is unchanged."""
    if before == after: return ""
    lines = difflib.unified_diff(
        str(before).splitlines(),str(after).splitlines(),
        fromfile=fromfile,tofile=tofile,lineterm="",n=context)
    return chr(10).join(lines)

In [ ]:

#| export
def insert_lines(text, insert_line, new_lines):
    """Insert `new_lines` before a 1-based line boundary in `text`; use n+1 to append."""
    lines = str(text).splitlines()
    try: idx = int(insert_line)
    except (TypeError, ValueError) as err: raise ValueError("insert_line must be an integer") from err
    if idx < 0: idx = len(lines) + idx + 2
    if idx < 1 or idx > len(lines) + 1: raise ValueError(f"insert_line must be within 1:{len(lines) + 1}; got {insert_line!r}")
    return _join_lines([*lines[:idx - 1], *_coerce_lines(new_lines), *lines[idx - 1:]])

In [ ]:

#| export
def replace_lines(text, start_line, end_line=None, replacement_lines=None):
    """Replace a 1-based inclusive line range in `text`."""
    lines = str(text).splitlines()
    start, end = _norm_lines(text, start_line, end_line)
    return _join_lines([*lines[:start], *_coerce_lines(replacement_lines), *lines[end:]])

In [ ]:

#| export
def delete_lines(text, start_line=None, end_line=None, re_filter=None, invert_filter=False):
    """Delete a 1-based inclusive line range, or lines matching `re_filter`."""
    lines = str(text).splitlines()
    if re_filter is not None:
        pattern = re.compile(re_filter)
        kept = []
        for line in lines:
            matched = bool(pattern.search(line))
            if invert_filter: matched = not matched
            if not matched: kept.append(line)
        return _join_lines(kept)
    if start_line is None: raise ValueError("delete_lines needs start_line unless re_filter is set")
    start, end = _norm_lines(text, start_line, end_line)
    return _join_lines([*lines[:start], *lines[end:]])

In [ ]:

#| export
def replace_text(text, old, new, start_line=None, end_line=None):
    """Replace literal `old` with `new`, optionally inside a line range."""
    if old in {None, ""}: raise ValueError("replace_text needs a non-empty old value")
    source = str(text)
    if start_line is None and end_line is None: return source.replace(str(old), str(new))
    lines = source.splitlines()
    start, end = _norm_lines(source, start_line or 1, end_line)
    segment = _join_lines(lines[start:end]).replace(str(old), str(new))
    return _join_lines([*lines[:start], *segment.splitlines(), *lines[end:]])

In [ ]:

#| export
def _replacement_items(replacements=None, olds=None, news=None):
    if replacements is None:
        if olds is None or news is None: raise ValueError("replace_texts needs replacements or olds/news")
        if len(olds) != len(news): raise ValueError("olds and news must have the same length")
        replacements = [{"old": old, "new": new} for old, new in zip(olds, news)]
    return [dict(item) for item in replacements]

In [ ]:

#| export
def replace_texts(text, replacements=None, olds=None, news=None, start_line=None, end_line=None):
    """Apply several literal text replacements in order."""
    result = str(text)
    for item in _replacement_items(replacements, olds, news):
        result = replace_text(result, item.get("old"), item.get("new", ""), start_line=start_line, end_line=end_line)
    return result

In [ ]:

#| export
def _replacement_match_count(source, old, start_line=None, end_line=None):
    if old in {None, ""}: return 0
    source = str(source)
    if start_line is None and end_line is None: return source.count(str(old))
    lines = source.splitlines()
    start, end = _norm_lines(source, start_line or 1, end_line)
    return _join_lines(lines[start:end]).count(str(old))

In [ ]:

#| export
def _replacement_statuses(source, edit):
    op = edit["op"]
    if op == "replace_text": items = [{"old": edit.get("old"), "new": edit.get("new", "")}]
    elif op == "replace_texts": items = _replacement_items(edit.get("replacements"), edit.get("olds"), edit.get("news"))
    else: return []
    statuses, current = [], str(source)
    for item in items:
        old, new = item.get("old"), item.get("new", "")
        matched = _replacement_match_count(current, old, edit.get("start_line"), edit.get("end_line"))
        updated = replace_text(current, old, new, edit.get("start_line"), edit.get("end_line"))
        statuses.append({
            "old": str(old), "new": str(new), "matched": matched,
            "changed": updated != current, "not_found": matched == 0,
        })
        current = updated
    return statuses

The public helper functions `insert_lines`, `replace_lines`, `delete_lines`, `replace_text`, and `replace_texts` use one-based line bounds for `insert_line`, `start_line`, and `end_line`, plus the same text semantics that `edit_notebook` applies to notebook cells.

In [ ]:
sample = "alpha\nbeta\ngamma"

print(insert_lines(sample, 2, ["inserted"]))
print(replace_lines(sample, 2, 2, ["BETA"]))
print(delete_lines(sample, re_filter="beta"))
print(replace_text(sample, "alpha", "ALPHA"))
print(replace_texts(sample, [dict(old="alpha", new="A"), dict(old="gamma", new="G")]))

In [ ]:
#| hide
assert replace_text("alpha\nbeta", "beta", "BETA") == "alpha\nBETA"

In [ ]:
#| hide
assert _coerce_lines(["a", "", "b\n\nc"]) == ["a", "", "b", "", "c"]
assert insert_lines("a\nc", 2, ["b"]) == "a\nb\nc"
assert insert_lines("a\nb", 1, ["start"]) == "start\na\nb"
assert insert_lines("a\nb", 3, ["end"]) == "a\nb\nend"
assert replace_lines("a\nb\nc", 2, 2, ["B"]) == "a\nB\nc"
assert delete_lines("a\nb\nc", 2, 2) == "a\nc"
assert delete_lines("keep\ndrop", re_filter="drop") == "keep"
assert replace_text("one two one", "one", "1") == "1 two 1"
assert replace_texts("a b", [{"old": "a", "new": "A"}, {"old": "b", "new": "B"}]) == "A B"
statuses = _replacement_statuses("a b", {"op": "replace_texts", "replacements": [{"old": "a", "new": "A"}, {"old": "missing", "new": "M"}]})
assert statuses == [
    {"old": "a", "new": "A", "matched": 1, "changed": True, "not_found": False},
    {"old": "missing", "new": "M", "matched": 0, "changed": False, "not_found": True},
]
assert _unified_diff("", _join_lines(["alpha", "", "beta"])).splitlines()[2:] == ["@@ -0,0 +1,3 @@", "+alpha", "+", "+beta"]
assert _unified_diff("same", "same") == ""

## Notebook-aware edits

These operations add the notebook concerns: stable cell ids, metadata, output clearing, validation, and one atomic commit.

#### Notebook adapter

The adapter selects cells, applies one deterministic operation per selected cell, validates the edited notebook before writing, clears stale outputs for changed code cells, stamps metadata, exports with nbdev, and returns structured results. The whole call succeeds or fails as one unit.

In [ ]:
#| export
_TEXT_OPS = {"replace_lines", "insert_lines", "delete_lines", "replace_text", "replace_texts"}
_STRUCTURAL_OPS = {"replace_cell", "insert_cells", "delete_cells", "move_cells", "explode_cells"}
_EDIT_OPS = _TEXT_OPS | _STRUCTURAL_OPS

In [ ]:
#| export
def _cell_by_id_map(nb):
    return {getattr(cell, "id", None): (idx, cell) for idx, cell in enumerate(nb.cells)}

In [ ]:
#| export
def _require_op(edit):
    if not isinstance(edit, dict): raise ValueError("each edit must be a dict")
    op = edit.get("op")
    if op not in _EDIT_OPS: raise ValueError(f"unsupported edit op {op!r}")
    return op

In [ ]:
#| export
def _replacement_source(edit):
    if "source_lines" in edit: lines = edit["source_lines"]
    elif "source" in edit: lines = edit["source"]
    else: raise ValueError("replace_cell needs source_lines or source")
    source = _join_lines(_coerce_lines(lines))
    if source == "": raise ValueError("replace_cell cannot delete a cell; use delete_cells")
    return source

In [ ]:
#| export
def _cell_from_source(source, cell_type="code"):
    cell = mk_cell(source, cell_type=cell_type)
    clear_outputs(cell)
    return cell

In [ ]:
#| export
def _cells_from_structs(cells, default_cell_type="code", directive=None):
    if not cells: raise ValueError("insert_cells needs a non-empty cells list")
    made = []
    for spec in cells:
        spec = spec or {}
        cell_type = spec.get("cell_type", default_cell_type)
        source = _join_lines(_coerce_lines(spec.get("source_lines", spec.get("source", []))))
        if cell_type == "code": source = apply_directives(source, spec.get("directive", directive))
        made.append(_cell_from_source(source, cell_type=cell_type))
    return made

In [ ]:
#| export
def _node_start_line(node):
    starts = [node.lineno, *[decorator.lineno for decorator in getattr(node, "decorator_list", [])]]
    return min(starts) - 1

In [ ]:
#| export
def _with_directive_prefix(source, directive_lines):
    source = "\n".join(str(source or "").splitlines()).strip("\n")
    if not source or not directive_lines: return source
    existing = {line.strip() for line in source.splitlines()}
    missing = [line for line in directive_lines if line not in existing]
    return "\n".join([*missing, source]) if missing else source

In [ ]:
#| export
def _append_exploded_source(chunks, lines, directive_lines):
    source = "\n".join(lines).strip("\n")
    if source: chunks.append(_with_directive_prefix(source, directive_lines))

In [ ]:
#| export
def _explode_function_sources(source):
    raw = str(source or "").strip("\n")
    if not raw: return []
    directive_lines, body_lines = _cell_directive_body_lines(raw)
    body = "\n".join(body_lines).strip("\n")
    if not body: return [raw]
    try:
        tree = ast.parse(body)
    except SyntaxError:
        return [raw]
    if sum(1 for node in tree.body if _is_function_def(node)) <= 1:
        return [raw]

    chunks, pending, cursor = [], [], 0
    for node in tree.body:
        start = _node_start_line(node)
        end = node.end_lineno
        gap = body_lines[cursor:start]
        node_lines = body_lines[start:end]
        if _is_function_def(node):
            _append_exploded_source(chunks, pending, directive_lines)
            pending = []
            _append_exploded_source(chunks, [*gap, *node_lines], directive_lines)
        else:
            pending.extend([*gap, *node_lines])
        cursor = end
    pending.extend(body_lines[cursor:])
    _append_exploded_source(chunks, pending, directive_lines)
    return chunks or [raw]

In [ ]:
#| export
def _explode_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return [cell]
    sources = _explode_function_sources(cell_source(cell))
    if len(sources) <= 1: return [cell]
    new_cells = [_cell_from_source(source, cell_type="code") for source in sources]
    new_cells[0].id = getattr(cell, "id", new_cells[0].id)
    return new_cells

In [ ]:
#| export
def _check_expected_hash(nb, edit, indices):
    expected = edit.get("expected_hash")
    if expected is None: return
    if isinstance(expected, dict):
        for idx in indices:
            cell = nb.cells[idx]
            cell_id = getattr(cell, "id", "")
            want = expected.get(cell_id)
            if want is not None and want != source_hash(cell_source(cell)):
                raise ValueError(f"expected_hash mismatch for cell {cell_id}")
        return
    if len(indices) != 1: raise ValueError("single expected_hash only works with one selected cell")
    cell = nb.cells[indices[0]]
    if str(expected) != source_hash(cell_source(cell)):
        raise ValueError(f"expected_hash mismatch for cell {getattr(cell, 'id', '')}")

In [ ]:
#| export
def _cell_content_match(cell, edit):
    if edit.get("cell_type") and getattr(cell, "cell_type", None) != edit.get("cell_type"): return False
    checks = []
    source = cell_source(cell)
    if edit.get("contains") is not None: checks.append(str(edit["contains"]) in source)
    if edit.get("re_filter") is not None: checks.append(bool(re.search(str(edit["re_filter"]), source, re.MULTILINE)))
    matched = all(checks) if checks else True
    return not matched if edit.get("invert_filter") else matched

In [ ]:
#| export
def _selected_indices(nb, edit):
    ids = []
    if edit.get("cell_id") is not None: ids = [edit["cell_id"]]
    elif edit.get("cell_ids") is not None: ids = list(edit["cell_ids"])
    elif edit.get("target") == "all":
        return [idx for idx, cell in enumerate(nb.cells) if _cell_content_match(cell, edit)]
    else:
        raise ValueError(f"{edit.get('op')} needs cell_id, cell_ids, or target='all'")
    seen = _cell_by_id_map(nb)
    missing = [cell_id for cell_id in ids if cell_id not in seen]
    if missing: raise ValueError(f"unknown cell id(s): {missing}")
    return [seen[cell_id][0] for cell_id in ids if _cell_content_match(seen[cell_id][1], edit)]

In [ ]:
#| export
def _apply_explode_edit(nb, edit, diffs, affected):
    indices = sorted(_selected_indices(nb, edit))
    if not indices: raise ValueError("explode_cells matched no cells")
    _check_expected_hash(nb, edit, indices)
    shift = 0
    for original_idx in indices:
        idx = original_idx + shift
        cell = nb.cells[idx]
        before = cell_source(cell)
        new_cells = _explode_cell(cell)
        after = "\n---\n".join(cell_source(item) for item in new_cells)
        changed = len(new_cells) > 1
        exploded_ids = [getattr(item, "id", "") for item in new_cells]
        diffs.append({
            "op": "explode_cells",
            "cell_id": getattr(cell, "id", ""),
            "cell_type": getattr(cell, "cell_type", ""),
            "changed": changed,
            "matches": 1 if changed else 0,
            "exploded_cell_ids": exploded_ids if changed else [],
            "before_hash": source_hash(before),
            "after_hash": source_hash(after),
            "diff": _unified_diff(
                before,
                after,
                fromfile=f"{getattr(cell, 'id', '')}:before",
                tofile=f"{getattr(cell, 'id', '')}:after",
            ),
        })
        if not changed: continue
        nb.cells[idx:idx + 1] = new_cells
        affected.extend(exploded_ids)
        shift += len(new_cells) - 1

In [ ]:
#| export
def _edit_cell_source(source, edit):
    op = edit["op"]
    if op == "replace_lines":
        return replace_lines(source, edit.get("start_line"), edit.get("end_line"), edit.get("replacement_lines", []))
    if op == "insert_lines":
        return insert_lines(source, edit.get("insert_line", edit.get("start_line")), edit.get("new_lines", edit.get("source_lines", [])))
    if op == "delete_lines":
        return delete_lines(source, edit.get("start_line"), edit.get("end_line"), edit.get("line_filter"), bool(edit.get("invert_filter")))
    if op == "replace_text":
        return replace_text(source, edit.get("old"), edit.get("new", ""), edit.get("start_line"), edit.get("end_line"))
    if op == "replace_texts":
        return replace_texts(source, edit.get("replacements"), edit.get("olds"), edit.get("news"), edit.get("start_line"), edit.get("end_line"))
    raise ValueError(f"{op!r} is not a text operation")

In [ ]:
#| export
def _append_cell_diff(diffs, cell, before, after, op, replacements=None):
    diff = _unified_diff(before, after, fromfile=f"{getattr(cell, 'id', '')}:before", tofile=f"{getattr(cell, 'id', '')}:after")
    record = {
        "op": op,
        "cell_id": getattr(cell, "id", ""),
        "cell_type": getattr(cell, "cell_type", ""),
        "changed": before != after,
        "matches": 0 if before == after else 1,
        "before_hash": source_hash(before),
        "after_hash": source_hash(after),
        "diff": diff,
    }
    if replacements is not None: record["replacements"] = replacements
    diffs.append(record)

In [ ]:
#| export
def _short_replacement_value(value, limit=40):
    text = str(value).replace("\n", "\\n")
    return text if len(text) <= limit else text[:limit - 3] + "..."

In [ ]:
#| export
def _aggregate_replacement_summary(diffs):
    rows = {}
    for diff in diffs:
        for item in diff.get("replacements") or []:
            key = (item.get("old", ""), item.get("new", ""))
            row = rows.setdefault(key, {
                "old": key[0], "new": key[1], "matched": 0,
                "changed": 0, "not_found": 0,
            })
            row["matched"] += int(item.get("matched") or 0)
            row["changed"] += int(bool(item.get("changed")))
            row["not_found"] += int(bool(item.get("not_found")))
    return list(rows.values())

In [ ]:
#| export
def _format_replacement_summary(summary):
    if not summary: return ""
    lines = ["Replacement summary:", "old | new | matched | changed | not_found"]
    for row in summary:
        lines.append(
            f"{_short_replacement_value(row['old'])} | {_short_replacement_value(row['new'])} | "
            f"{row['matched']} | {row['changed']} | {row['not_found']}"
        )
    return "\n".join(lines)

In [ ]:
#| export
def _format_export_confirmation(path, commit):
    if not commit.get("exported"): return ""
    py_path = commit.get("exported_py_path") or ""
    if not py_path: return ""
    drift = "no drift remaining" if commit.get("export_no_drift") else "drift check unavailable"
    return f"exported {py_path} from {path}; {drift}"

In [ ]:
#| export
def _apply_text_edit(nb, edit, diffs, affected):
    indices = _selected_indices(nb, edit)
    _check_expected_hash(nb, edit, indices)
    for idx in indices:
        cell = nb.cells[idx]
        before = cell_source(cell)
        replacements = _replacement_statuses(before, edit) if edit["op"] in {"replace_text", "replace_texts"} else None
        after = _edit_cell_source(before, edit)
        _append_cell_diff(diffs, cell, before, after, edit["op"], replacements=replacements)
        if after != before:
            cell.source = after
            clear_outputs(cell)
            affected.append(getattr(cell, "id", ""))

fastcore 2.0 now provides the raw notebook insertion and movement mechanics through `fastcore.nbio.Notebook`. nbskill still builds cells, applies directives, validates code, records hashes, and formats MCP diffs here; only the low-level list mutation is delegated.

In [ ]:
#| export
def _insert_at(nb, anchor_id, where, new_cells):
    if where not in {"before", "after"}: raise ValueError("where must be 'before' or 'after'")
    model = FastcoreNotebook(nb)
    inserted, last_id = [], None
    for cell in new_cells:
        kwargs = {"cell_type": getattr(cell, "cell_type", "code")}
        if last_id is not None:
            kwargs["after"] = last_id
        elif anchor_id is not None:
            kwargs[where] = anchor_id
        elif where == "before" and len(model):
            kwargs["before"] = model[0].id
        added = model.add(cell_source(cell), **kwargs)
        added.id = getattr(cell, "id", added.id)
        clear_outputs(added)
        inserted.append(getattr(added, "id", ""))
        last_id = getattr(added, "id", "")
    return inserted

In [ ]:
#| export
def _apply_structural_edit(nb, edit, diffs, affected, default_cell_type):
    op = edit["op"]
    if op == "replace_cell":
        indices = _selected_indices(nb, edit)
        if len(indices) != 1: raise ValueError("replace_cell needs exactly one selected cell")
        _check_expected_hash(nb, edit, indices)
        cell = nb.cells[indices[0]]
        before = cell_source(cell)
        after = _replacement_source(edit)
        cell_type = edit.get("cell_type", getattr(cell, "cell_type", default_cell_type))
        if cell_type == "code": after = apply_directives(after, edit.get("directive"))
        new_cell = _cell_from_source(after, cell_type=cell_type)
        new_cell.id = getattr(cell, "id", new_cell.id)
        _append_cell_diff(diffs, cell, before, after, op)
        if after != before or getattr(cell, "cell_type", None) != cell_type:
            nb.cells[indices[0]] = new_cell
            affected.append(getattr(new_cell, "id", ""))
        return
    if op == "insert_cells":
        new_cells = _cells_from_structs(edit.get("cells"), default_cell_type=default_cell_type, directive=edit.get("directive"))
        inserted = _insert_at(nb, edit.get("anchor_id"), edit.get("where", "after"), new_cells)
        affected.extend(inserted)
        inserted_source = "\n---\n".join(cell_source(cell) for cell in new_cells)
        diffs.append({
            "op": op,
            "cell_id": "",
            "changed": True,
            "inserted_cell_ids": inserted,
            "before_hash": "",
            "after_hash": source_hash(inserted_source),
            "diff": _unified_diff("", inserted_source, fromfile="insert:before", tofile="insert:after"),
        })
        return
    if op == "delete_cells":
        indices = _selected_indices(nb, edit)
        if not indices: raise ValueError("delete_cells matched no cells")
        _check_expected_hash(nb, edit, indices)
        for idx in sorted(indices, reverse=True):
            cell = nb.cells[idx]
            affected.append(getattr(cell, "id", ""))
            diffs.append({"op": op, "cell_id": getattr(cell, "id", ""), "changed": True, "before_hash": source_hash(cell_source(cell)), "after_hash": "", "diff": _unified_diff(cell_source(cell), "")})
            del nb.cells[idx]
        return
    if op == "explode_cells":
        _apply_explode_edit(nb, edit, diffs, affected)
        return
    if op == "move_cells":
        indices = _selected_indices(nb, edit)
        if not indices: raise ValueError("move_cells matched no cells")
        before_order = [getattr(cell, "id", "") for cell in nb.cells]
        moving = [nb.cells[idx] for idx in indices]
        moving_ids = [getattr(cell, "id", "") for cell in moving]
        anchor_id = edit.get("anchor_id")
        where = edit.get("where", "after")
        if where not in {"before", "after"}: raise ValueError("where must be 'before' or 'after'")
        try:
            FastcoreNotebook(nb).move(moving_ids, **{where: anchor_id})
        except KeyError as err:
            raise ValueError(f"unknown anchor_id {anchor_id!r}") from err
        after_order = [getattr(cell, "id", "") for cell in nb.cells]
        affected.extend(moving_ids)
        diff = "\n".join([
            f"moved cells: {', '.join(moving_ids)}",
            f"anchor: {where} {anchor_id}",
        ])
        diffs.append({
            "op": op, "cell_id": "", "changed": True, "moved_cell_ids": moving_ids,
            "anchor_id": anchor_id, "where": where, "before_order": before_order,
            "after_order": after_order, "diff": diff,
        })
        return
    raise ValueError(f"unsupported structural op {op!r}")

In [ ]:
#| export
def edit_notebook(
    path,
    edits,
    validate_code=True,
    default_cell_type="code",
    directive=None,
    auto_feedback=True,
    feedback_timeout=10,
    feedback_safe=True,
    detail="summary",
    dry_run=False,
):
    """Apply structured notebook edits atomically and return MCP-friendly details."""
    if not edits: raise ValueError("edits must be a non-empty list")
    path = Path(path)
    normalized = [dict(edit) for edit in edits]
    _validate_edit_budget(normalized)
    for edit in normalized:
        _require_op(edit)
        if directive is not None: edit.setdefault("directive", directive)

    with notebook_locks(path):
        before = read_nb(path)
        trial = copy.deepcopy(before)
        diffs, affected = [], []
        for edit in normalized:
            op = edit["op"]
            if op in _TEXT_OPS: _apply_text_edit(trial, edit, diffs, affected)
            else: _apply_structural_edit(trial, edit, diffs, affected, default_cell_type)
        affected = [cell_id for cell_id in dict.fromkeys(affected) if cell_id]
        if len(affected) > _EDIT_MAX_AFFECTED_CELLS:
            raise ValueError(
                f"edit_notebook budget exceeded: {len(affected)} affected cells exceeds limit {_EDIT_MAX_AFFECTED_CELLS}"
            )
        warnings = _edit_notebook_craft_warnings(path, trial, affected)
        commit = commit_notebook(
            path,
            trial,
            before=before,
            affected_cell_ids=affected,
            validate_code=validate_code,
            dry_run=dry_run,
        )

    changed = commit["changed"]
    diffs = _cap_edit_diffs(diffs)
    changed_diffs = [item for item in diffs if item.get("changed")]
    replacement_summary = _aggregate_replacement_summary(diffs)
    export_confirmation = _format_export_confirmation(path, commit)
    status = "dry_run" if dry_run and changed else ("changed" if changed else "no_change")
    text = f"edit_notebook {status}: {len(changed_diffs)} changed operation(s), {len(affected)} affected cell(s)"
    warning_text = _format_edit_warnings(warnings)
    if warning_text: text = f"{text}\n\nNotebook craft warnings:\n{warning_text}"
    replacement_text = _format_replacement_summary(replacement_summary)
    if replacement_text: text = f"{text}\n\n{replacement_text}"
    if export_confirmation: text = f"{text}\n{export_confirmation}"
    diff_text = "\n\n".join(item["diff"] for item in changed_diffs if item.get("diff"))
    diff_text = cap_text(diff_text, _EDIT_MAX_DIFF_CHARS)
    if diff_text: text = f"{text}\n\n{diff_text}"
    if changed and not dry_run:
        text = append_notebook_edit_feedback(
            text, path, affected, auto_feedback=auto_feedback, feedback_timeout=feedback_timeout, feedback_safe=feedback_safe
        )
    return {
        "ok": True,
        "changed": changed,
        "no_change": not changed,
        "dry_run": dry_run,
        "affected_cell_ids": affected,
        "diffs": diffs,
        "before_hash": commit["before_hash"],
        "after_hash": commit["after_hash"],
        "planned_hash": commit["planned_hash"],
        "readback_hashes": commit["readback_hashes"],
        "exported": commit["exported"],
        "exported_py_path": commit.get("exported_py_path", ""),
        "export_no_drift": commit.get("export_no_drift", False),
        "export_confirmation": export_confirmation,
        "replacement_summary": replacement_summary,
        "warnings": warnings,
        "text": text,
    }

## Editor facade

The facade keeps repeated edits scoped to one notebook while the lower-level functions remain useful for one-off transformations.

#### A notebook-scoped editor

`NotebookEditor` is a small facade over `edit_notebook` for one notebook, and `Notebook.edit` hands one back. Edits accept a `directive` (a `Directive`, its name, or a comma string such as `export,no_test`) so callers can say `export and do not test` instead of typing `#|` lines. The directive is stamped onto code cells, and context strips it back out because the semantic type already conveys it.

In [ ]:
#| export
@dataclass
class NotebookEditor:
    'Notebook-scoped facade over edit_notebook.'
    path: object
    validate_code: bool = True
    auto_feedback: bool = False
    def __post_init__(self): self.path = str(self.path)

    def apply(self, *edits, **kw):
        'Apply one or more structured edit ops atomically.'
        kw.setdefault('validate_code', self.validate_code)
        kw.setdefault('auto_feedback', self.auto_feedback)
        return edit_notebook(self.path, list(edits), **kw)

In [ ]:
#| export
@patch
def replace_cell(self:NotebookEditor, cell_id, source, directive=None, cell_type=None, **kw):
    'Replace one cell source, optionally stamping directives onto code.'
    edit = {'op': 'replace_cell', 'cell_id': cell_id, 'source': source}
    if directive is not None: edit['directive'] = directive
    if cell_type is not None: edit['cell_type'] = cell_type
    return self.apply(edit, **kw)

In [ ]:
#| export
@patch
def insert(self:NotebookEditor, anchor_id, source, where='after', directive=None, cell_type='code', **kw):
    'Insert one cell relative to anchor_id; pass None to append.'
    cell = {'source': source, 'cell_type': cell_type}
    if directive is not None: cell['directive'] = directive
    return self.apply({'op': 'insert_cells', 'anchor_id': anchor_id, 'where': where, 'cells': [cell]}, **kw)

In [ ]:
#| export
@patch
def explode(self:NotebookEditor, cell_id=None, **kw):
    'Split top-level function definitions in one cell, or matching cells, into separate cells.'
    edit = {'op': 'explode_cells'}
    if cell_id is None: edit['target'] = 'all'
    else: edit['cell_id'] = cell_id
    return self.apply(edit, **kw)

In [ ]:
#| export
@patch
def delete(self:NotebookEditor, cell_id, **kw):
    'Delete one cell by id.'
    return self.apply({'op': 'delete_cells', 'cell_id': cell_id}, **kw)

In [ ]:
#| export
@patch
def replace_text(self:NotebookEditor, old, new='', cell_id=None, **kw):
    'Replace literal text in one cell, or across the notebook when cell_id is omitted.'
    edit = {'op': 'replace_text', 'old': old, 'new': new}
    if cell_id is None: edit['target'] = 'all'
    else: edit['cell_id'] = cell_id
    return self.apply(edit, **kw)

In [ ]:
#| export
@patch
def edit(self: Notebook):
    'Return a NotebookEditor bound to this notebook path.'
    return NotebookEditor(self.path)

In [ ]:
#| hide
from fastcore.test import *
import nbskill.foundation as foundation
from nbskill.foundation import write_demo_notebook
from contextlib import contextmanager
from fastcore.nbio import new_nb, write_nb
from tempfile import gettempdir

In [ ]:

with write_demo_notebook('editor_directive_demo.ipynb', cells=[mk_cell('def helper():\n    return 1')]) as demo:
    editor = Notebook.from_path(demo).edit()
    print(type(editor).__name__)
    cell_id = read_nb(demo).cells[0].id
    editor.replace_cell(cell_id, 'def helper():\n    return 2', directive='export,no_test')
    print(read_nb(demo).cells[0].source)

In [ ]:
#| hide
with write_demo_notebook('editor_directive_test.ipynb', cells=[mk_cell('value = 1'), mk_cell('## Note', cell_type='markdown')]) as demo:
    nb = Notebook.from_path(demo)
    assert isinstance(nb.edit(), NotebookEditor)
    assert nb.edit().path == str(demo)
    code_id, md_id = read_nb(demo).cells[0].id, read_nb(demo).cells[1].id
    res = nb.edit().replace_cell(code_id, 'value = 2', directive='export,no_test')
    assert res['changed']
    src = read_nb(demo).cells[0].source
    assert src.startswith('#| export\n#| eval: false\n') and 'value = 2' in src
    nb.edit().replace_cell(md_id, '## Note', directive='export', cell_type='markdown')
    assert '#|' not in read_nb(demo).cells[1].source

A notebook-wide rename is just a `replace_text` operation with `target="all"`. That path is the default for deterministic refactors such as renaming a public symbol across Markdown, examples, and tests.

In [ ]:

with write_demo_notebook("edit_notebook_example.ipynb", cells=[mk_cell("old_name = 1")]) as example_path:
    result = edit_notebook(
        example_path,
        [dict(op="replace_text", target="all", old="old_name", new="new_name")],
        auto_feedback=False,
    )
    source = read_nb(example_path).cells[0].source
source

### Hash-guarded dry runs

An `expected_hash` protects the exact cell source selected by `context(..., mode="edit")`. Use a dry run to inspect the one-cell diff and replacement count before writing. In the MCP tool, `feedback="off"` suppresses automatic feedback; smoke snippets remain an advanced, explicit option.

In [ ]:
with write_demo_notebook("02_edit_hash_example.ipynb", cells=[mk_cell("value = 1")]) as path:
    cell = read_nb(path).cells[0]
    result = edit_notebook(path, [dict(op="replace_text", cell_id=cell.id, old="1", new="2", expected_hash=source_hash(cell_source(cell)))], auto_feedback=False, dry_run=True)
    print(result["affected_cell_ids"], result["replacement_summary"])

In [ ]:
#| hide
@contextmanager
def _outside_edit_probe():
    outside = Path(gettempdir()) / "nbskill_edit_outside_export_probe.ipynb"
    outside_py = Path("nbskill/_outside_edit_probe.py")
    outside.unlink(missing_ok=True)
    outside_py.unlink(missing_ok=True)
    outside_nb = new_nb([mk_cell("#| default_exp _outside_edit_probe"), mk_cell("probe = 1")])
    write_nb(outside_nb, outside)
    try:
        yield outside, outside_py, outside_nb
    finally:
        outside.unlink(missing_ok=True)
        outside_py.unlink(missing_ok=True)

In [ ]:
#| hide
with write_demo_notebook("edit_notebook_demo.ipynb", cells=[mk_cell("value = 1"), mk_cell("value")]) as demo:
    nb = read_nb(demo)
    cell_id = nb.cells[0].id
    assert callable(edit_notebook)
    result = edit_notebook(
        demo,
        [{"op": "replace_lines", "cell_id": cell_id, "start_line": 1, "end_line": 1, "replacement_lines": ["value = 2"]}],
        auto_feedback=False,
    )
    assert result["ok"] and result["changed"]
    assert "---" in result["text"]
    assert "exported with nbdev" not in result["text"]
    assert read_nb(demo).cells[0].source == "value = 2"

    replacements_result = edit_notebook(
        demo,
        [{
            "op": "replace_texts",
            "cell_id": cell_id,
            "replacements": [
                {"old": "value", "new": "number"},
                {"old": "absent", "new": "present"},
            ],
        }],
        auto_feedback=False,
        dry_run=True,
    )
    assert "Replacement summary:" in replacements_result["text"]
    assert replacements_result["replacement_summary"] == [
        {"old": "value", "new": "number", "matched": 1, "changed": 1, "not_found": 0},
        {"old": "absent", "new": "present", "matched": 0, "changed": 0, "not_found": 1},
    ]

    dry_result = edit_notebook(
        demo,
        [{"op": "replace_text", "cell_id": cell_id, "old": "2", "new": "3"}],
        auto_feedback=False,
        dry_run=True,
    )
    assert dry_result["ok"] and dry_result["changed"] and dry_result["dry_run"]
    assert dry_result["affected_cell_ids"] == [cell_id]
    assert "---" in dry_result["text"]
    assert dry_result["before_hash"] == dry_result["after_hash"]
    assert dry_result["planned_hash"] != dry_result["before_hash"]
    assert cell_id in dry_result["readback_hashes"]
    assert read_nb(demo).cells[0].source == "value = 2"

    bad_hash = [{"op": "replace_text", "cell_id": cell_id, "old": "2", "new": "3", "expected_hash": "nope"}]
    test_fail(edit_notebook,args=(demo, bad_hash),kwargs=dict(auto_feedback=False),exc=ValueError,contains="expected_hash")
    test_eq(read_nb(demo).cells[0].source, "value = 2")

In [ ]:
#| hide
with write_demo_notebook(
    "edit_notebook_move_summary.ipynb",
    cells=[mk_cell("a = 1"), mk_cell("b = 2"), mk_cell("c = 3")],
) as demo:
    nb = read_nb(demo)
    before_order = [cell.id for cell in nb.cells]
    moving_id = nb.cells[2].id
    anchor_id = nb.cells[0].id
    result = edit_notebook(
        demo,
        [dict(op="move_cells", cell_ids=[moving_id], anchor_id=anchor_id, where="before")],
        auto_feedback=False,
    )
    move_diff = result["diffs"][0]
    test_eq(move_diff["diff"], f"moved cells: {moving_id}\nanchor: before {anchor_id}")
    test_eq(move_diff["moved_cell_ids"], [moving_id])
    test_eq(move_diff["anchor_id"], anchor_id)
    test_eq(move_diff["where"], "before")
    test_eq(move_diff["before_order"], before_order)
    test_eq(move_diff["after_order"], [moving_id, *before_order[:2]])
    test_eq([cell.id for cell in read_nb(demo).cells], move_diff["after_order"])
    assert "before order:" not in result["text"]
    assert "after order:" not in result["text"]

In [ ]:
#| hide
with write_demo_notebook(
    "edit_notebook_fastcore_structural.ipynb",
    cells=[mk_cell("a = 1"), mk_cell("b = 2")],
) as demo:
    nb = read_nb(demo)
    first_id, second_id = nb.cells[0].id, nb.cells[1].id
    insert_result = edit_notebook(
        demo,
        [dict(
            op="insert_cells", anchor_id=first_id, where="before",
            cells=[dict(cell_type="markdown", source="## Head"), dict(source="inserted = True")],
        )],
        auto_feedback=False,
    )
    inserted_ids = insert_result["affected_cell_ids"]
    nb = read_nb(demo)
    assert ([cell.source for cell in nb.cells[:3]], FastcoreNotebook(nb).view(inserted_ids[1], nums=False)) == (["## Head", "inserted = True", "a = 1"], "inserted = True")
    move_result = edit_notebook(
        demo,
        [dict(op="move_cells", cell_ids=[inserted_ids[1]], anchor_id=second_id, where="after")],
        auto_feedback=False,
    )
    nb = read_nb(demo)
    assert ([cell.id for cell in nb.cells][-1], move_result["diffs"][0]["moved_cell_ids"]) == (inserted_ids[1], [inserted_ids[1]])

In [ ]:
#| hide
with write_demo_notebook(
    "edit_notebook_rollback.ipynb",
    cells=[mk_cell("#| default_exp edit_rollback_probe"), mk_cell("#| export\nvalue = 1")],
    base="nbs",
) as rollback_nb:
    before_bytes = rollback_nb.read_bytes()
    rollback_cell_id = read_nb(rollback_nb).cells[1].id
    old_export = foundation.run_nbdev_export_from_project
    try:
        def fail_export(path): raise RuntimeError("forced export failure")
        foundation.run_nbdev_export_from_project = fail_export
        try:
            edit_notebook(
                rollback_nb,
                [{"op": "replace_text", "cell_id": rollback_cell_id, "old": "1", "new": "2"}],
                auto_feedback=False,
            )
            raise AssertionError("export failure should fail")
        except RuntimeError as err:
            assert "forced export failure" in str(err)
    finally:
        foundation.run_nbdev_export_from_project = old_export
    assert rollback_nb.read_bytes() == before_bytes
    assert read_nb(rollback_nb).cells[1].source == "#| export\nvalue = 1"

with _outside_edit_probe() as (outside, outside_py, outside_nb):
    outside_result = edit_notebook(
        outside,
        [dict(op="replace_text", cell_id=outside_nb.cells[1].id, old="1", new="2")],
        auto_feedback=False,
    )
    assert outside_result["changed"] and not outside_result["exported"]
    assert not outside_py.exists()

In [ ]:
#| hide
with write_demo_notebook("edit_notebook_append.ipynb", cells=[]) as append_path:
    result = NotebookEditor(append_path).insert(None, "created = 1", auto_feedback=False)
    nb = read_nb(append_path)
    assert result["changed"] is True
    assert len(nb.cells)
    assert nb.cells[3].source == "created = 1"

In [ ]:
#| hide
with write_demo_notebook(
    "edit_notebook_explode.ipynb",
    cells=[mk_cell("#| export\ndef first():\n    return 1\n\ndef second():\n    return 2")],
) as demo:
    nb = read_nb(demo)
    original_id = nb.cells[0].id
    result = edit_notebook(demo, [dict(op="explode_cells", cell_id=original_id)], auto_feedback=False)
    exploded = read_nb(demo)
    assert result["changed"]
    assert len(exploded.cells) == 2
    assert exploded.cells[0].id == original_id
    assert all(cell.source.startswith("#| export\n") for cell in exploded.cells)
    assert [cell.source.splitlines()[1] for cell in exploded.cells] == ["def first():", "def second():"]
    assert all("nbskill" in cell.metadata for cell in exploded.cells)

In [ ]:
#| hide
with write_demo_notebook(
    "edit_notebook_craft_multi_function.ipynb",
    cells=[mk_cell("#| export\ndef old():\n    return 0")],
) as demo:
    cell_id = read_nb(demo).cells[0].id
    source = "#| export\n" + "\n\n".join(f"def fn_{i}():\n    return {i}" for i in range(4))
    result = edit_notebook(demo, [dict(op="replace_cell", cell_id=cell_id, source=source)], auto_feedback=False)
    warnings = result["warnings"]
    assert any(warning["code"] == "multi-function-cell" for warning in warnings)
    assert any("op=\"explode_cells\"" in warning["message"] and cell_id in warning["message"] for warning in warnings)

with write_demo_notebook(
    "edit_notebook_craft_large_cell.ipynb",
    cells=[mk_cell("seed = 1")],
) as demo:
    cell_id = read_nb(demo).cells[0].id
    large_source = "\n".join(f"value_{i} = {i}" for i in range(_EDIT_WARN_CODE_LINES + 1))
    result = edit_notebook(
        demo,
        [dict(op="insert_cells", anchor_id=cell_id, cells=[dict(cell_type="code", source=large_source)])],
        auto_feedback=False,
    )
    assert [warning["code"] for warning in result["warnings"]] == ["large-code-cell"]
    assert len(read_nb(demo).cells) == 2

with write_demo_notebook(
    "edit_notebook_craft_small_replace.ipynb",
    cells=[mk_cell("value = 1")],
) as demo:
    cell_id = read_nb(demo).cells[0].id
    result = edit_notebook(
        demo,
        [dict(op="replace_lines", cell_id=cell_id, start_line=1, end_line=1, replacement_lines=["value = 2"])],
        auto_feedback=False,
    )
    assert result["warnings"] == []

In [ ]:
#| hide
with write_demo_notebook(
    "edit_notebook_craft_order.ipynb",
    cells=[mk_cell("result = ready()"), mk_cell("def later_helper():\n    return 1")],
) as demo:
    cell_id = read_nb(demo).cells[0].id
    result = edit_notebook(
        demo,
        [dict(op="replace_text", cell_id=cell_id, old="ready", new="later_helper")],
        auto_feedback=False,
    )
    warning = next(item for item in result["warnings"] if item["code"] == "cell-order")
    assert (warning["cell_id"], warning["symbol"], "later_helper" in warning["message"], "cell-order" in result["text"]) == (cell_id, "later_helper", True, True)

In [ ]:
#| hide
with write_demo_notebook(
    "edit_notebook_export_confirm.ipynb",
    cells=[mk_cell("#| default_exp edit_export_confirm"), mk_cell("#| export\nvalue = 1")],
    base="nbs",
) as export_demo:
    export_cell_id = read_nb(export_demo).cells[1].id
    export_result = edit_notebook(
        export_demo,
        [dict(op="replace_text", cell_id=export_cell_id, old="1", new="2")],
        auto_feedback=False,
    )
    try:
        assert not export_result["exported"]
        assert export_result["exported_py_path"].endswith("nbskill/edit_export_confirm.py")
        assert export_result["export_confirmation"] == ""
        assert "exported " not in export_result["text"]
    finally:
        exported = export_result.get("exported_py_path")
        if exported: Path(exported).unlink(missing_ok=True)